In [ ]:
import sys
import os

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add parent directories to path
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'parameters'))
sys.path.insert(0, os.getcwd())

from analysis_utils import (
    load_best_runs,
    load_top_n_runs,
    extract_history,
    add_wall_times,
    plot_2d_histograms,
    plot_training_curves,
    plot_time_to_best,
    plot_speedrun_results,
    print_speedrun_table,
    print_best_hyperparameters,
    save_best_hyperparameters,
    get_optimizer_colors,
    ALL_OPTIMIZERS,
)

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")


# Configuration


In [ ]:
# Backend: "wandb" or "local"
BACKEND = "local"

# WandB settings (ignored if BACKEND == "local")
PROJECT = "induced_metric"
ENTITY = "thomas_harvey"

# Task settings
TASK_TAG = "regression"
ITERATION = 0  # Iteration/batch number to load results from

# Optimizers to analyse
OPTIMIZERS = ['adam', 'adamw', 'sgd', 'sgd_metric', 'sgd_log_metric', 'muon', 'sgd_rms']
# OPTIMIZERS = ALL_OPTIMIZERS  # Uncomment to analyse all optimizers

# Results directory (for local backend)
RESULTS_DIR = os.path.join('..', 'results')

# Metric configuration
SORT_METRIC = "final_min_val_loss"   # WandB summary metric to sort by
SORT_ORDER = "+"                      # "+" ascending (lower loss = better)
METRIC_KEY = "sweep_metric"
DIRECTION = "minimize"

# Speedrun targets (test MSE thresholds)
SPEEDRUN_TARGETS = [0.5, 0.1, 0.05, 0.02, 0.01, 0.005, 0.002, 0.001]
SPEEDRUN_DIRECTION = "below"
SPEEDRUN_METRIC = "test_mse"

# History metrics to extract
HISTORY_METRICS = ["train_loss", "train_mse", "test_mse", "train_time_seconds"]

# 2D histogram axes
HIST_X = "final_min_loss_epoch"
HIST_Y = "final_min_val_loss"

# Time-to-best keys
BEST_EPOCH_KEY = "final_min_loss_epoch"
BEST_METRIC_KEY = "test_mse"

colors = get_optimizer_colors(OPTIMIZERS)
print(f"Backend: {BACKEND}")
print(f"Task: {TASK_TAG}, Iteration: {ITERATION}")
print(f"Optimizers: {OPTIMIZERS}")

# Load Data


In [ ]:
best_runs = load_best_runs(
    backend=BACKEND,
    optimizers=OPTIMIZERS,
    task_tag=TASK_TAG,
    project=PROJECT,
    entity=ENTITY,
    results_dir=RESULTS_DIR,
    metric_key=METRIC_KEY,
    direction=DIRECTION,
    sort_metric=SORT_METRIC,
    sort_order=SORT_ORDER,
    iteration=ITERATION,
)

top_n_runs = load_top_n_runs(
    backend=BACKEND,
    optimizers=OPTIMIZERS,
    n=50,
    task_tag=TASK_TAG,
    project=PROJECT,
    entity=ENTITY,
    results_dir=RESULTS_DIR,
    metric_key=METRIC_KEY,
    direction=DIRECTION,
    sort_metric=SORT_METRIC,
    sort_order=SORT_ORDER,
    iteration=ITERATION,
)

print(f"\nLoaded best runs for {len(best_runs)} optimizers")

# Distribution of Top Runs


In [ ]:
fig = plot_2d_histograms(
    top_n_runs, backend=BACKEND,
    x_metric=HIST_X, y_metric=HIST_Y,
    optimizers=OPTIMIZERS,
)
plt.show()


# Training History


In [ ]:
optimizer_data = extract_history(BACKEND, best_runs, HISTORY_METRICS)
add_wall_times(optimizer_data)

print(f"Extracted history for {len(optimizer_data)} optimizers")


In [ ]:
fig = plot_time_to_best(
    best_runs, optimizer_data,
    best_epoch_key=BEST_EPOCH_KEY,
    best_metric_key=BEST_METRIC_KEY,
    colors=colors,
)
plt.show()


# Training Curves


In [ ]:
fig = plot_training_curves(
    optimizer_data,
    x_keys=["wall_times", "epoch"],
    y_keys=["train_loss", "test_mse"],
    colors=colors,
)
plt.show()


# Speedrun Analysis


In [ ]:
print("Speedrun Results: Time (seconds) to reach target test MSE")
print("=" * 80)
print_speedrun_table(optimizer_data, SPEEDRUN_TARGETS, SPEEDRUN_METRIC, SPEEDRUN_DIRECTION)

fig = plot_speedrun_results(
    optimizer_data, SPEEDRUN_TARGETS,
    metric_key=SPEEDRUN_METRIC,
    direction=SPEEDRUN_DIRECTION,
    colors=colors,
)
plt.show()


# Best Hyperparameters


In [ ]:
print_best_hyperparameters(best_runs)

filename = f'best_hyperparameters_Regression_itr_{ITERATION}.csv'
df = save_best_hyperparameters(best_runs, filename)
df